In [5]:
import pandas as pd
import numpy as np
from scipy.io import loadmat
from pathlib import Path
import mne

In [20]:
data_folder = Path('../Data/')
raw_data_folder = Path('../Data/raw_data/')

In [ ]:
data_labels = pd.read_csv(data_folder / "preprocessed_data.csv")

In [13]:
data_labels.head()

,subject,trial,kind,file_name,stress_level
0,1,1,Relax,Relax_sub_1_trial1.mat,0
1,1,1,Mirror,Mirror_image_sub_1_trial1.mat,3
2,1,1,Arithmetic,Arithmetic_sub_1_trial1.mat,6
3,1,1,Stroop,Stroop_sub_1_trial1.mat,3
4,1,2,Mirror,Mirror_image_sub_1_trial2.mat,5


In [14]:
data_labels.shape

(480, 5)

In [15]:
data_labels["kind"].value_counts()

kind
Relax         120
Mirror        120
Arithmetic    120
Stroop        120
Name: count, dtype: int64

In [18]:
stress_df = data_labels[data_labels["kind"] != "Relax"].copy()

In [19]:
stress_df["kind"].value_counts()

kind
Mirror        120
Arithmetic    120
Stroop        120
Name: count, dtype: int64

## process each file

In [25]:
SFREQ = 128
N_CHANNELS = 32

LOW_FREQ = 0.5
HIGH_FREQ = 45

In [39]:
def process_file(filepath: str):
    mat = loadmat(filepath)
    data = mat['Data'] # data.shape => (32, 3200) -> 32 channels, 3200 samples

    ch_names = [
        f"EEG{i:03d}"
        for i in range(1, N_CHANNELS + 1)
    ]


    info = mne.create_info(
        ch_names=ch_names,
        sfreq=SFREQ,
        ch_types="eeg"
    )

    raw = mne.io.RawArray(
        data,
        info,
        verbose=False
    )

    raw_filtered = raw.copy()

    raw_filtered.filter(
        l_freq=LOW_FREQ,
        h_freq=HIGH_FREQ,
        verbose=False
    )


    ica = mne.preprocessing.ICA(
        n_components=15,
        random_state=42,
        max_iter=1000,
        method="picard",
        fit_params={"tol": 0.01}
    )

    ica.fit(
        raw_filtered,
        verbose=False,
    )


    raw_clean = raw_filtered.copy()


    ica.apply(
        raw_clean,
        verbose=False
    )

    epochs = mne.make_fixed_length_epochs(
        raw_clean,
        duration=2.0,
        overlap=1.0,
        preload=True,
        verbose=False
    )

    X = epochs.get_data()
    file_path = str(filepath).split("/")[-1]
    stress_level = data_labels[data_labels["file_name"] == file_path]["stress_level"].values[0]

    y = np.full(
        X.shape[0],
        stress_level
    )
    print(y)
    return X

In [40]:
process_file(raw_data_folder / "Mirror_image_sub_1_trial1.mat")

[3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3]


array([[[-1.74224753e-15,  4.35017869e+00,  2.70943834e+00, ...,
          3.19999562e+01,  2.69097024e+01,  1.70553907e+01],
        [-2.71338921e-14,  4.67381502e+00,  5.71265399e+00, ...,
          1.28917586e+01,  5.61351667e+00, -3.40799115e+00],
        [-2.42624100e-14,  4.74409545e+00,  1.07013416e+01, ...,
         -4.23078009e-01, -6.66977292e+00, -1.31508136e+01],
        ...,
        [-7.35615624e-15,  6.70104903e+00,  1.04962232e+01, ...,
          2.07998948e+01,  1.59785078e+01,  8.74748693e+00],
        [-4.64599341e-15,  2.81539326e+00,  5.16127783e+00, ...,
          1.89985659e+01,  1.52381388e+01,  9.06695967e+00],
        [-2.50367423e-14, -9.49967057e-01, -8.42467173e-01, ...,
          2.84399909e+01,  2.13899062e+01,  1.82693530e+01]],

       [[ 5.38958198e+00,  3.20608738e+00,  5.21774257e+00, ...,
          2.63203117e+01,  1.49122956e+01,  4.61348630e+00],
        [-8.91829135e+00, -5.07901106e+00, -2.21242496e+00, ...,
         -1.59367659e+01, -2.30442329e